# Road Accident Severity CNN — CCTV Dataset

Train a three-class EfficientNetV2-S image classifier on the [Road Accidents from CCTV Footages dataset](https://www.kaggle.com/datasets/suryaprabhakaran2005/road-accidents-from-cctv-footages-dataset).

Class mapping: `NonAccident` → `No_Accident`; severity `[1]` → `Minor_Accident`; severity `[2]` + `[3]` → `Severe_Accident`. The generic `Accident` folder is excluded because it overlaps the severity subsets. The dataset publisher reports 15,420 non-accident, 1,874 minor, 2,100 substantial and 2,217 critical images; counts are checked after scanning the downloaded files.

**Important:** the Kaggle listing says MIT, but its source CCTV dataset lists its license as unknown. Check the original terms before redistribution/commercial use. The archive's score spreadsheets identify source clips, but do not map each flattened image filename to a clip. Supply `data/source_video_groups.csv` with `relative_path,group_id` for a true source-video split. Otherwise this notebook groups consecutive numeric frame filenames in blocks of 100 to reduce adjacent-frame leakage; this is only a proxy, not a verified video split.


## 1. Prepare the data

The preferred option is the manually downloaded `archive.zip` in this folder. This cell safely extracts only the mapped class images and severity metadata; it skips the overlapping generic `Accident` images. If the archive is absent, it falls back to the Kaggle CLI.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
from pathlib import PurePosixPath
from zipfile import ZipFile
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd()
DATA_ROOT = PROJECT_DIR / "data" / "raw"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
mapped_roots = [
    DATA_ROOT / "NonAccident",
    DATA_ROOT / "SeverityScore" / "Severity Score Dataset with Labels" / "1",
    DATA_ROOT / "SeverityScore" / "Severity Score Dataset with Labels" / "2",
    DATA_ROOT / "SeverityScore" / "Severity Score Dataset with Labels" / "3",
]
already_downloaded = all(
    root.exists() and any(p.is_file() and p.suffix.lower() in image_extensions for p in root.rglob("*"))
    for root in mapped_roots
)
local_archive = PROJECT_DIR / "archive.zip"

if not already_downloaded:
    if local_archive.is_file():
        with ZipFile(local_archive) as archive:
            selected = []
            for member in archive.infolist():
                member_path = PurePosixPath(member.filename)
                if member.is_dir() or not member_path.parts:
                    continue
                if member_path.is_absolute() or ".." in member_path.parts or ":" in member_path.parts[0]:
                    raise ValueError(f"Unsafe path in dataset archive: {member.filename}")
                is_class_data = member_path.parts[0] in {"NonAccident", "SeverityScore"}
                is_image = member_path.suffix.lower() in image_extensions
                is_severity_metadata = member_path.parts[0] == "SeverityScore" and member_path.suffix.lower() == ".xlsx"
                if is_class_data and (is_image or is_severity_metadata):
                    selected.append(member)
            image_members = [m for m in selected if PurePosixPath(m.filename).suffix.lower() in image_extensions]
            if not image_members:
                raise ValueError(f"No mapped image files found in {local_archive}")
            required_bytes = sum(member.file_size for member in selected)
            if shutil.disk_usage(DATA_ROOT).free < required_bytes:
                raise OSError(f"Not enough free disk space to extract {len(image_members):,} images from {local_archive}")
            for member in tqdm(selected, desc="Extracting mapped dataset files"):
                destination = DATA_ROOT.joinpath(*PurePosixPath(member.filename).parts)
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, destination.open("wb") as target:
                    shutil.copyfileobj(source, target)
        print(f"Extracted {len(image_members):,} mapped images from {local_archive.name}; skipped generic Accident duplicates.")
    else:
        try:
            subprocess.run(
                [sys.executable, "-m", "kaggle", "datasets", "download",
                 "-d", "suryaprabhakaran2005/road-accidents-from-cctv-footages-dataset",
                 "-p", str(DATA_ROOT), "--unzip"],
                check=True,
            )
        except FileNotFoundError as exc:
            raise RuntimeError("Place archive.zip in this project folder, or install the Kaggle CLI and configure Kaggle API credentials.") from exc
        except subprocess.CalledProcessError as exc:
            raise RuntimeError("Kaggle download failed. Confirm Kaggle credentials and that you accepted the dataset terms.") from exc
else:
    print(f"Using existing image files under {DATA_ROOT}")

print(f"Dataset root: {DATA_ROOT}")


## 2. Imports, reproducibility, and hardware

Install this subproject's `requirements.txt` before running if any import is missing. Install a GPU-compatible PyTorch build when needed.

In [ ]:
import random
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, roc_auc_score, roc_curve, auc)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["No_Accident", "Minor_Accident", "Severe_Accident"]
CLASS_TO_IDX = {name: index for index, name in enumerate(CLASS_NAMES)}
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 25
PATIENCE = 6
NUM_WORKERS = 0  # Safe default on Windows.
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Project: {PROJECT_DIR}")


## 3. Build the image index and apply the three-class mapping

Only `NonAccident` and the numbered `SeverityScore` folders are used. The broad `Accident` folder is deliberately not treated as an extra class.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def normalize_folder_name(value):
    return re.sub(r"[^a-z0-9]", "", value.lower())

def label_for_path(path):
    parts = [part.lower().strip() for part in path.relative_to(DATA_ROOT).parts[:-1]]
    normalized = [normalize_folder_name(part) for part in parts]
    if any(part in {"nonaccident", "noaccident"} for part in normalized):
        return "No_Accident"

    in_severity_folder = any("severityscore" in part for part in normalized)
    if in_severity_folder:
        severity_number = next((part.strip("[]() ") for part in reversed(parts)
                                if part.strip("[]() ").isdigit()), None)
        if severity_number == "1":
            return "Minor_Accident"
        if severity_number in {"2", "3"}:
            return "Severe_Accident"
    return None

records = []
for image_path in DATA_ROOT.rglob("*"):
    if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
        label = label_for_path(image_path)
        if label is not None:
            records.append({
                "path": str(image_path.resolve()),
                "relative_path": image_path.relative_to(DATA_ROOT).as_posix(),
                "label": label,
            })

manifest = pd.DataFrame(records).drop_duplicates(subset="path").reset_index(drop=True)
if manifest.empty:
    raise FileNotFoundError(
        f"No supported class images found under {DATA_ROOT}. Check that the Kaggle archive was extracted and the folder layout is intact."
    )
missing = sorted(set(CLASS_NAMES) - set(manifest["label"]))
if missing:
    raise ValueError(f"Dataset is missing mapped classes {missing}; found: {manifest['label'].value_counts().to_dict()}")

class_counts = manifest["label"].value_counts().reindex(CLASS_NAMES, fill_value=0)
display(class_counts.rename("images").to_frame())
print(f"Total mapped images: {len(manifest):,}")
print("The published description reports 21,611 images across these mapped classes; counts above are the files actually found.")


## 4. Create and save reproducible splits

The preferred option is `data/source_video_groups.csv` with one row per image and columns `relative_path,group_id`, using the actual source video for each image. The supplied archive does not include that per-image mapping, so the fallback groups numeric filenames in consecutive 100-frame blocks within each class. This reduces neighboring-frame leakage, but does not guarantee true video-independent splits. Do not tune the split to hit a target accuracy: report the held-out results from this fixed split.

In [ ]:
group_map_path = PROJECT_DIR / "data" / "source_video_groups.csv"
if group_map_path.exists():
    group_map = pd.read_csv(group_map_path, dtype={"relative_path": str, "group_id": str})
    required = {"relative_path", "group_id"}
    if not required.issubset(group_map.columns):
        raise ValueError(f"{group_map_path} must contain columns {sorted(required)}")
    if group_map["relative_path"].duplicated().any() or group_map["group_id"].isna().any():
        raise ValueError("Video-group mapping has duplicate image paths or missing group IDs.")
    manifest = manifest.merge(group_map[list(required)], on="relative_path", how="left", validate="one_to_one")
    if manifest["group_id"].isna().any():
        missing_paths = manifest.loc[manifest["group_id"].isna(), "relative_path"].head(5).tolist()
        raise ValueError(f"Video-group mapping is missing images (examples): {missing_paths}")
    split_mode = "source-video grouped"
else:
    FRAME_BLOCK_SIZE = 100
    frame_numbers = pd.to_numeric(manifest["relative_path"].map(lambda path: Path(path).stem), errors="coerce")
    if frame_numbers.isna().any():
        examples = manifest.loc[frame_numbers.isna(), "relative_path"].head(5).tolist()
        raise ValueError(
            "No source-video mapping was supplied and some image filenames are not numeric, "
            f"so contiguous-frame grouping is unavailable (examples: {examples})."
        )
    manifest["group_id"] = (
        manifest["label"].astype(str) + ":" +
        (frame_numbers // FRAME_BLOCK_SIZE).astype(int).astype(str).str.zfill(5)
    )
    split_mode = f"contiguous {FRAME_BLOCK_SIZE}-frame blocks (video-ID proxy)"
    print(
        "WARNING: the archive does not map each image to its source video. "
        "Grouping contiguous numeric filenames reduces adjacent-frame leakage but is only a proxy."
    )

grouped_splits = None
for attempt in range(100):
    first_split = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED + attempt)
    train_val_idx, test_idx = next(first_split.split(manifest, y=manifest["label"], groups=manifest["group_id"]))
    candidate_train_val = manifest.iloc[train_val_idx].copy()
    candidate_test = manifest.iloc[test_idx].copy()
    second_split = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=SEED + attempt)
    train_idx, val_idx = next(second_split.split(
        candidate_train_val, y=candidate_train_val["label"], groups=candidate_train_val["group_id"]
    ))
    candidate_train = candidate_train_val.iloc[train_idx].copy()
    candidate_val = candidate_train_val.iloc[val_idx].copy()
    if all(set(frame["label"]) == set(CLASS_NAMES)
           for frame in (candidate_train, candidate_val, candidate_test)):
        grouped_splits = (candidate_train, candidate_val, candidate_test)
        break
if grouped_splits is None:
    raise ValueError(
        "Could not make group-disjoint train/validation/test splits containing all classes. "
        "Check the video-group mapping and ensure each class spans enough source videos."
    )
train, val, test = grouped_splits

for split_name, split_frame in [("train", train), ("validation", val), ("test", test)]:
    absent = sorted(set(CLASS_NAMES) - set(split_frame["label"]))
    if absent:
        raise ValueError(f"{split_name} split is missing classes {absent}; adjust the group mapping/split strategy.")
    split_frame["split"] = split_name

split_manifest = pd.concat([train, val, test], ignore_index=True)
split_manifest.to_csv(
    PROJECT_DIR / "data" / "split_manifest.csv",
    index=False,
    columns=["relative_path", "label", "group_id", "split"],
)
print(f"Split mode: {split_mode}")
display(pd.crosstab(split_manifest["split"], split_manifest["label"]).reindex(columns=CLASS_NAMES, fill_value=0))
if "group_id" in split_manifest.columns:
    group_sets = {name: set(frame["group_id"]) for name, frame in [("train", train), ("validation", val), ("test", test)]}
    assert not (group_sets["train"] & group_sets["validation"])
    assert not (group_sets["train"] & group_sets["test"])
    assert not (group_sets["validation"] & group_sets["test"])


## 5. Image transforms and data loaders

In [ ]:
from torchvision import transforms

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class AccidentImageDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["path"]) as image:
            image = image.convert("RGB")
        return self.transform(image), CLASS_TO_IDX[row["label"]]

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
}
train_ds = AccidentImageDataset(train, train_transform)
val_ds = AccidentImageDataset(val, eval_transform)
test_ds = AccidentImageDataset(test, eval_transform)
train_loader = DataLoader(train_ds, shuffle=True, **loader_options)
val_loader = DataLoader(val_ds, shuffle=False, **loader_options)
test_loader = DataLoader(test_ds, shuffle=False, **loader_options)
print(f"Train: {len(train_ds):,} | validation: {len(val_ds):,} | test: {len(test_ds):,}")


## 6. Initialize the pretrained CNN

The first run downloads pretrained ImageNet weights if they are not already cached.

In [ ]:
model = timm.create_model("tf_efficientnetv2_s", pretrained=True, num_classes=len(CLASS_NAMES))
model = model.to(DEVICE)

train_counts = train["label"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_numpy(dtype=np.float32)
if np.any(train_counts == 0):
    raise ValueError(f"Cannot train: missing a class in train split: {dict(zip(CLASS_NAMES, train_counts))}")
class_weights = len(train) / (len(CLASS_NAMES) * train_counts)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Class weights: {dict(zip(CLASS_NAMES, class_weights.round(3)))}")


## 7. Train with validation-based checkpointing and early stopping

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for images, labels in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += batch_size
    return total_loss / total_samples, total_correct / total_samples

history = []
best_val_accuracy = -1.0
epochs_without_improvement = 0
checkpoint_path = ARTIFACT_DIR / "best_model.pth"

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader, training=False)
    scheduler.step(val_accuracy)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "learning_rate": optimizer.param_groups[0]["lr"],
    })
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0
        torch.save({
            "model_state": model.state_dict(),
            "class_names": CLASS_NAMES,
            "epoch": epoch,
            "val_accuracy": val_accuracy,
            "image_size": IMAGE_SIZE,
            "model_name": "tf_efficientnetv2_s",
        }, checkpoint_path)
        marker = " *"
    else:
        epochs_without_improvement += 1
        marker = ""
    print(f"{epoch:02d}/{NUM_EPOCHS} train_loss={train_loss:.4f} train_acc={train_accuracy:.3%} "
          f"val_loss={val_loss:.4f} val_acc={val_accuracy:.3%} lr={optimizer.param_groups[0]['lr']:.2e}{marker}")
    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {epoch} epochs.")
        break

history = pd.DataFrame(history)
history.to_csv(ARTIFACT_DIR / "training_history.csv", index=False)
print(f"Best validation accuracy: {best_val_accuracy:.2%}")
print(f"Checkpoint saved to: {checkpoint_path}")


## 8. Evaluate once on the held-out test split

The test split is not used for model selection. Review per-class recall/F1 as well as overall accuracy because the classes are imbalanced.

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model_state"])
model.eval()
all_labels, all_predictions, all_probabilities = [], [], []
with torch.inference_mode():
    for images, labels in tqdm(test_loader, desc="Test evaluation"):
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(images)
        probabilities = torch.softmax(logits.float(), dim=1)
        all_labels.extend(labels.numpy().tolist())
        all_predictions.extend(probabilities.argmax(dim=1).cpu().numpy().tolist())
        all_probabilities.extend(probabilities.cpu().numpy().tolist())

all_labels = np.asarray(all_labels)
all_predictions = np.asarray(all_predictions)
all_probabilities = np.asarray(all_probabilities)
print(f"Best checkpoint: epoch {checkpoint['epoch']} | validation accuracy {checkpoint['val_accuracy']:.2%}")
print(f"Test accuracy: {accuracy_score(all_labels, all_predictions):.2%}")
print(f"Test weighted F1: {f1_score(all_labels, all_predictions, average='weighted'):.4f}")
print(classification_report(all_labels, all_predictions, labels=range(len(CLASS_NAMES)),
                            target_names=CLASS_NAMES, zero_division=0))
test_probabilities_df = pd.DataFrame(all_probabilities, columns=[f"prob_{name}" for name in CLASS_NAMES])
test_export = pd.concat([
    test.reset_index(drop=True)[["relative_path", "label"]].rename(columns={"label": "true"}),
    pd.Series([CLASS_NAMES[index] for index in all_predictions], name="pred"),
    test_probabilities_df,
], axis=1)
test_export.to_csv(ARTIFACT_DIR / "test_predictions.csv", index=False)


## 9. Save training curves, confusion matrix, and ROC curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], label="Train")
axes[0].plot(history["epoch"], history["val_loss"], label="Validation")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[0].legend()
axes[1].plot(history["epoch"], history["train_accuracy"], label="Train")
axes[1].plot(history["epoch"], history["val_accuracy"], label="Validation")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
axes[1].legend()
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "training_curves.png", dpi=160)
plt.show()

cm = confusion_matrix(all_labels, all_predictions, labels=range(len(CLASS_NAMES)))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set(title="Test confusion matrix", xlabel="Predicted", ylabel="Actual")
sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt=".1%", cmap="Greens",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1], vmin=0, vmax=1)
axes[1].set(title="Row-normalized confusion matrix", xlabel="Predicted", ylabel="Actual")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for index, class_name in enumerate(CLASS_NAMES):
    binary_labels = (all_labels == index).astype(int)
    fpr, tpr, _ = roc_curve(binary_labels, all_probabilities[:, index])
    class_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{class_name} (AUC={class_auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="Test one-vs-rest ROC")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "roc_curves.png", dpi=160)
plt.show()
print(f"Macro one-vs-rest ROC AUC: {roc_auc_score(all_labels, all_probabilities, multi_class='ovr', average='macro'):.4f}")


## 10. Predict an image

Set `sample_path` to an image path and run the cell to get the class probabilities.

In [ ]:
def predict_image(image_path):
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")
    with Image.open(image_path) as image:
        image = image.convert("RGB")
    tensor = eval_transform(image).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                                                enabled=DEVICE.type == "cuda"):
        probabilities = torch.softmax(model(tensor).float(), dim=1)[0].cpu().numpy()
    return {name: float(probability) for name, probability in zip(CLASS_NAMES, probabilities)}

sample_path = test.iloc[0]["path"]
print(sample_path)
print(predict_image(sample_path))
